In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install -U albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.9/269.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 632.7/632.7 kB 32.9 MB/s eta 0:00:00
  Attempting uninstall: albucore
    Found existing installation: albucore 0.0.19
    Uninstalling albucore-0.0.19:
      Successfully uninstalled albucore-0.0.19
  Attempting uninstall: albumentations
    Found existing installation: albumentations 1.4.20
    Uninstalling albumentations-1.4.20:
      Successfully uninstalled albumentations-1.4.20


In [ ]:
!git clone https://github.com/ultralytics/ultralytics.git
!cd ultralytics && pip install .

Cloning into 'ultralytics'...
remote: Enumerating objects: 45724, done.
remote: Counting objects: 100% (675/675), done.
remote: Compressing objects: 100% (393/393), done.
remote: Total 45724 (delta 551), reused 300 (delta 281), pack-reused 45049 (from 4)
Receiving objects: 100% (45724/45724), 38.92 MiB | 17.41 MiB/s, done.
Resolving deltas: 100% (33884/33884), done.
Processing /content/ultralytics
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.51-py3-none-any.whl size=901417 sha256=b17b336fa5b19ed6c5c15322eafa8b8a505739b2c463cfc1d90aeb3fa9b6f8ec
  Stored in directory: /tmp/pip-ephem-wheel-cache-qullfthe/wheels/9a/cd/d5/95912172899f8ec640166ff6eef49156b1b00d6b2ade4a3cb1
Successfully built ultralytics


In [ ]:
import yaml
from ultralytics import YOLO
from google.colab import files, drive
import os

# Mount Google Drive to save weights
drive.mount('/content/drive')

# Paths to the unzipped dataset
train_images_path = '/content/drive/MyDrive/UAV/FOGGY/split_data/images/train'
val_images_path = '//content/drive/MyDrive/UAV/FOGGY/split_data/images/val'
train_labels_path = '/content/drive/MyDrive/UAV/FOGGY/split_data/labels/train'
val_labels_path = '/content/drive/MyDrive/UAV/FOGGY/split_data/labels/val'

# Create a temporary dictionary for data configuration
data_config = {
    'train': train_images_path,  # Path to training images
    'val': val_images_path,      # Path to validation images
    'nc': 35,                    # Number of classes
    'names': [                   # Class names
        "Traffic Signal", "Lamp Post", "Zebra Crossing", "Bike", "Car", "Rikshaw", "Tyre Works",
        "Tree", "Tractor", "Cattle", "Vegetation", "Electricity Pole", "Building", "Board", "Wall",
        "Person", "Bus", "Bridge", "Road Divider", "Tempo", "Traffic Sign Board", "Flag", "Crane",
        "Cycle", "Dog", "Truck", "Glove", "Overbridge", "Manhole", "Bus Stop", "Barricade",
        "Petrol Pump", "Ambulance", "Goat", "Cart"
    ]
}

# Save the configuration to a temporary file
config_file_path = '/content/drive/MyDrive/UAV/FOGGY/split_data (1).yaml'
with open(config_file_path, 'w') as file:
    yaml.dump(data_config, file)

print(f"Config file saved at {config_file_path}")

# Upload the weights file 'last.pt' to continue training
print("Please upload the weights file.")
uploaded = files.upload()  # Upload the file

# Print the names of the uploaded files for debugging
print("Uploaded files:", uploaded.keys())

# Ensure the uploaded file is correctly handled
uploaded_files = list(uploaded.keys())
if len(uploaded_files) == 0:
    raise FileNotFoundError("No file uploaded. Please upload the correct weights file.")
uploaded_file_name = uploaded_files[0]

# Handle file names like 'last (8).pt', 'last (9).pt', etc.
if 'last.pt' not in uploaded_file_name:
    # Rename the uploaded file to 'last.pt' if it contains 'last' and a number
    new_file_name = 'last.pt'
    os.rename(f'/content/{uploaded_file_name}', f'/content/{new_file_name}')
    uploaded_file_name = new_file_name
    print(f"Renamed the uploaded file to: {uploaded_file_name}")

# Path to the uploaded weights file
last_weights_path = f'/content/{uploaded_file_name}'
print(f"Using uploaded file: {uploaded_file_name}")

# Load the model from the uploaded weights
model = YOLO(last_weights_path)

# Resume training for 5 epochs or until 50 epochs total
total_epochs = 50
current_epoch = model.ckpt['epoch'] + 1  # Add 1 to the epoch because YOLOv8 starts from 0
remaining_epochs = total_epochs - current_epoch
epochs_to_train = min(35, remaining_epochs)  # Train for 5 or fewer epochs if near the target

if remaining_epochs > 0:
    print(f"Training for {epochs_to_train} epochs (Current Epoch: {current_epoch}).")
    model.train(data=config_file_path, epochs=current_epoch + epochs_to_train, batch=5)

    # Update the current epoch after training
    current_epoch += epochs_to_train

    # Save the updated weights
    new_weights_path = f'/content/drive/MyDrive/updated_weights_epoch_{current_epoch}.pt'
    model.save(new_weights_path)
    print(f"Updated weights saved at {new_weights_path}.")
else:
    print("Training is already complete. Total 50 epochs reached!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config file saved at /content/drive/MyDrive/UAV/FOGGY/split_data (1).yaml
Please upload the weights file.


Saving last.pt to last.pt
Uploaded files: dict_keys(['last.pt'])
Using uploaded file: last.pt
Training for 35 epochs (Current Epoch: 0).
Ultralytics 8.3.51 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
engine/trainer: task=detect, mode=train, model=/content/last.pt, data=/content/drive/MyDrive/UAV/FOGGY/split_data (1).yaml, epochs=35, time=None, patience=100, batch=5, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, cl

train: Scanning /content/drive/.shortcut-targets-by-id/1tgzt_EPjk6gqwUzCpoGaSJbqwilGDs07/UAV/FOGGY/split_data/labels/train.cache... 1041 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1041/1041 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Scanning /content/drive/.shortcut-targets-by-id/1tgzt_EPjk6gqwUzCpoGaSJbqwilGDs07/UAV/FOGGY/split_data/labels/val.cache... 151 images, 6 backgrounds, 0 corrupt: 100%|██████████| 157/157 [00:00<?, ?it/s]


...

      35/35      0.77G       1.89      2.267      1.771          1        640: 100%|██████████| 209/209 [05:08<00:00,  1.48s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 16/16 [00:37<00:00,  2.36s/it]

                   all        157        861      0.427      0.207      0.152     0.0666



35 epochs completed in 3.530 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 6.2MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.51 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
YOLOv8n summary (fused): 168 layers, 3,012,473 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 16/16 [00:34<00:00,  2.13s/it]


                   all        157        861      0.477       0.15      0.161     0.0701
        Traffic Signal         45         84      0.262      0.274      0.215     0.0966
             Lamp Post         28         68      0.258     0.0882     0.0784     0.0274
        Zebra Crossing         10         13      0.143      0.154       0.14     0.0691
                  Bike         81        171      0.351      0.643      0.462      0.181
                   Car         78        152      0.346       0.52      0.412        0.2
               Rikshaw         40         61      0.252      0.215      0.192      0.098
            Tyre Works          4         14      0.135      0.357      0.162     0.0702
                  Tree         26         40      0.315      0.175      0.181     0.0842
               Tractor          5          5          1          0     0.0215    0.00648
                Cattle          4         12          1          0     0.0182    0.00529
            Vegetatio

In [ ]:
import shutil
from google.colab import files

# Specify the source directory and the target ZIP file path
source_dir = "/content/runs"
output_zip = "/content/runs.zip"

# Create a ZIP file from the source directory
shutil.make_archive(output_zip.replace('.zip', ''), 'zip', source_dir)

# Download the ZIP file to the local system
files.download(output_zip)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>